# SHAP export for sharing (TrunCat, corrected CV)

Rebuilds `shap_values_for_sharing.pkl` from the **corrected** CV pipeline (gene-grouped folds, fixed iterations, no early stopping). Replaces cells 0-2 of the old `SHAP_summary.ipynb`.

Fixes relative to the old notebook:
1. **Fold membership is recovered from the saved OOF predictions**, not re-derived with `StratifiedKFold`. Each fold model reproduces its own out-of-fold predictions exactly, so every row is matched to the one model that never trained on it. This works for any splitter (including `StratifiedGroupKFold`).
2. **Exactly one model file per fold is required.** The old `glob(...)[0]` could silently pick a stale pre-fix model.
3. **Features come from `model.feature_names_`**, so the new `key` column in `TOPMed_cleaned.csv` cannot leak in as a feature.
4. SHAP additivity is checked per fold (SHAP sum + base value == model raw score), and a manifest is written next to the pickle.

Sampling (500 held-out variants per fold, `random_state=42`) and the pickle keys match the old file, plus a few extra keys (`sample_keys`, `sample_meta`, ...) that existing plotting notebooks ignore.

**Run from:** `Model/TrunCat/notebooks/`

In [ ]:
import glob, json, pickle, re, subprocess, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import shap
import catboost
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

NOTEBOOK_DIR = Path.cwd().resolve()      # .../Model/TrunCat/notebooks
TRUNCAT_DIR  = NOTEBOOK_DIR.parent       # .../Model/TrunCat
REPO_ROOT    = TRUNCAT_DIR.parent.parent # repo root

with open(TRUNCAT_DIR / "config" / "config.yaml") as f:
    cfg = yaml.safe_load(f)

def resolve(p):
    """Config paths are not all anchored the same way; take whichever base has the file."""
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    for base in (NOTEBOOK_DIR, TRUNCAT_DIR, REPO_ROOT):
        cand = (base / p).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(f"Could not resolve {p} relative to {NOTEBOOK_DIR}, {TRUNCAT_DIR} or {REPO_ROOT}")

# ---- inputs (all from config) ------------------------------------------------
TARGET        = cfg["model"]["target"]
N_FOLDS       = cfg["model"]["n_folds"]
CLEANED_PATH  = resolve(cfg["data"]["cleaned"])
CV_MODELS_DIR = resolve(cfg["output"]["cv_models_dir"])
CV_PRED_PATH  = resolve(cfg["output"]["cv_predictions"])   # must contain key, y_true, oof_prob

# ---- settings kept identical to the old export -------------------------------
RANDOM_STATE     = 42     # SHAP sampling seed AND jitter seed read by the plotting notebooks
SHAP_SAMPLE_SIZE = None    # held-out variants explained per fold

# ---- tripwires: values confirmed for the corrected pipeline ------------------
EXPECTED_TREES     = 250
EXPECTED_FOLD_AUCS = [0.7747, 0.7700, 0.8045, 0.7714, 0.7674]
EXPECTED_OOF_AUC   = 0.7760

# ---- output ------------------------------------------------------------------
OUTPUT_DIR = TRUNCAT_DIR / "results" / "figures" / "visualizations_cv" / "shap_manuscript"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PKL_PATH = OUTPUT_DIR / "shap_values_for_sharing.pkl"

print("cleaned data :", CLEANED_PATH)
print("fold models  :", CV_MODELS_DIR)
print("CV predictions:", CV_PRED_PATH)
print("output       :", PKL_PATH)

## 1. Load data, fold models, and OOF predictions

In [ ]:
df = pd.read_csv(CLEANED_PATH)
cv = pd.read_csv(CV_PRED_PATH)
cv = cv.rename(columns={"true_label": "y_true", "predicted_prob": "oof_prob"})

assert "key" in df.columns, "cleaned CSV has no `key` column (notebook 02 change missing?)"
assert {"key", "y_true", "oof_prob"} <= set(cv.columns), f"CV predictions missing columns: {list(cv.columns)}"
assert df["key"].is_unique and cv["key"].is_unique, "duplicate keys"

# --- exactly one model per fold, with the expected fold AUC in the filename ----
fold_models, fold_files = [], []
for k in range(1, N_FOLDS + 1):
    matches = sorted(glob.glob(str(CV_MODELS_DIR / f"fold_{k}_auc_*.cbm")))
    if len(matches) != 1:
        raise RuntimeError(
            f"Fold {k}: expected exactly 1 model file, found {len(matches)}: {matches}. "
            "If old pre-fix models are still in this folder, move them out first.")
    auc_in_name = float(re.search(r"auc_([0-9.]+)\.cbm", matches[0]).group(1))
    assert abs(auc_in_name - EXPECTED_FOLD_AUCS[k - 1]) < 5e-4, \
        f"Fold {k} file {Path(matches[0]).name}: AUC {auc_in_name} != expected {EXPECTED_FOLD_AUCS[k-1]} (stale model?)"
    m = CatBoostClassifier()
    m.load_model(matches[0])
    assert m.tree_count_ == EXPECTED_TREES, f"Fold {k}: tree_count_={m.tree_count_}, expected {EXPECTED_TREES}"
    fold_models.append(m)
    fold_files.append(Path(matches[0]).name)
    print(f"  fold {k}: {Path(matches[0]).name}  trees={m.tree_count_}")

# --- feature matrix: model.feature_names_ is the source of truth -------------
feature_names = list(fold_models[0].feature_names_)
assert all(list(m.feature_names_) == feature_names for m in fold_models), "fold models disagree on features"
assert "key" not in feature_names and TARGET not in feature_names, "identifier/target leaked into model features"
X = df[feature_names].copy()
y = df[TARGET].astype(int).to_numpy()
keys = df["key"].to_numpy()
cat_idx = list(fold_models[0].get_cat_feature_indices())
print(f"\nfeatures: {len(feature_names)} ({len(cat_idx)} categorical), samples: {len(X)}")

# --- OOF predictions aligned to the cleaned CSV by key ------------------------
cv_i = cv.set_index("key")
oof = cv_i["oof_prob"].reindex(keys).to_numpy()
assert not np.isnan(oof).any() and len(oof) == len(df), "some variants have no OOF prediction"
assert (cv_i["y_true"].reindex(keys).to_numpy() == y).all(), "y_true in CV predictions disagrees with cleaned CSV"
oof_auc = roc_auc_score(y, oof)
print(f"pooled OOF AUC from CV predictions: {oof_auc:.4f}")
assert abs(oof_auc - EXPECTED_OOF_AUC) < 5e-4, "CV predictions are not the corrected run"

## 2. Recover which fold each variant was held out in

A fold model reproduces its own out-of-fold predictions exactly, but not those of rows it trained on. So each row is assigned to the one fold model that reproduces its saved OOF probability (rows that two models both reproduce are excluded from sampling).

In [ ]:
def recover_fold_membership(models, X, cat_idx, oof, atol=1e-5):
    """Assign each row to the fold model whose prediction reproduces its saved OOF probability.

    Returns (fold_of_row, ambiguous). A row is ambiguous if a second fold model also
    reproduces its OOF value within atol (practically never happens with real models).
    """
    pool = Pool(X, cat_features=cat_idx)
    diff = np.column_stack([np.abs(m.predict_proba(pool)[:, 1] - oof) for m in models])
    fold_of_row = diff.argmin(axis=1)
    best = diff.min(axis=1)
    if not (best < atol).all():
        raise RuntimeError(
            f"{int((best >= atol).sum())} rows are not reproduced by any fold model "
            f"(worst diff {best.max():.2e}). The fold models on disk are not the ones that "
            "produced the CV predictions, or the feature dtypes differ from training.")
    ambiguous = (diff < atol).sum(axis=1) > 1
    return fold_of_row, ambiguous

fold_of_row, ambiguous = recover_fold_membership(fold_models, X, cat_idx, oof)
print(f"rows matched to a fold: {len(fold_of_row)}; ambiguous (excluded from SHAP sampling): {int(ambiguous.sum())}")
assert ambiguous.mean() < 0.005, "too many ambiguous rows; fold recovery is unreliable"

print("fold  n_test  fold_AUC(recovered)  fold_AUC(filename)")
for k in range(N_FOLDS):
    msk = fold_of_row == k
    a = roc_auc_score(y[msk], oof[msk])
    a_name = float(re.search(r"auc_([0-9.]+)\.cbm", fold_files[k]).group(1))
    assert abs(a - a_name) < 1e-3, f"fold {k+1}: recovered AUC {a:.4f} != filename AUC {a_name:.4f}"
    print(f"  {k+1}   {msk.sum():5d}   {a:.4f}              {a_name:.4f}")

## 3. SHAP values on 500 held-out variants per fold

In [ ]:
def compute_shap_export(models, X, keys, y, oof, fold_of_row, ambiguous, cat_idx, n_per_fold, seed):
    shap_parts, x_parts, meta_parts, base_vals = [], [], [], []
    for k, model in enumerate(models):
        idx = np.where((fold_of_row == k) & ~ambiguous)[0]
        Xk = X.iloc[idx]
        if n_per_fold is not None and len(Xk) > n_per_fold:
            Xk = Xk.sample(n=n_per_fold, random_state=seed)

        explainer = shap.TreeExplainer(model)
        sv = explainer.shap_values(Xk)
        if isinstance(sv, list):
            sv = sv[1]
            base = explainer.expected_value[1]
        else:
            base = explainer.expected_value
        base = float(np.ravel(base)[0])

        # additivity: SHAP contributions + base value must equal the model's raw (log-odds) score
        raw = model.predict(Pool(Xk, cat_features=cat_idx), prediction_type="RawFormulaVal")
        err = float(np.abs(sv.sum(axis=1) + base - raw).max())
        assert err < 1e-3, f"fold {k+1}: SHAP additivity error {err:.2e}"

        pos = Xk.index.to_numpy()
        shap_parts.append(sv)
        x_parts.append(Xk)
        base_vals.append(base)
        meta_parts.append(pd.DataFrame({
            "key": keys[pos], "fold": k + 1, "y_true": y[pos], "oof_prob": oof[pos]}))
        print(f"  fold {k+1}: {len(Xk)} samples, base value {base:.4f}, additivity err {err:.1e}")

    return (np.vstack(shap_parts),
            pd.concat(x_parts, ignore_index=True),
            pd.concat(meta_parts, ignore_index=True),
            base_vals)

shap_values_oof, X_shap_oof, sample_meta, base_values = compute_shap_export(
    fold_models, X, keys, y, oof, fold_of_row, ambiguous, cat_idx, SHAP_SAMPLE_SIZE, RANDOM_STATE)
base_value_avg = float(np.mean(base_values))

print(f"\nSHAP values: {shap_values_oof.shape}, feature data: {X_shap_oof.shape}, mean base value: {base_value_avg:.4f}")
assert sample_meta["key"].is_unique, "a variant was sampled twice"

## 4. Plot metadata (kept identical to the old export)

Color maps and display names are copied unchanged from `SHAP_summary.ipynb`.

In [ ]:
categorical_color_maps = {
    'last.EJC': {
        'last.exon':             '#e74c3c',
        'upstream':              '#3498db',
        'penultimate.last50bp':  '#2ecc71',
        'MISSING':               '#95a5a6'
    },
    'fiveutrseqs.uORF': {True: '#9b59b6', False: '#f39c12', 'MISSING': '#95a5a6'},
    'long.exon':        {True: '#9b59b6', False: '#f39c12', 'MISSING': '#95a5a6'},
    'first.200': {'first 200': '#9b59b6', 'not first 200': '#f39c12', 'MISSING': '#95a5a6'},
    'first.100': {'first 100': '#9b59b6', 'not first 100': '#f39c12', 'MISSING': '#95a5a6'},
}

feature_display_names = {
    'last.EJC':                    'last.EJC',
    'relativePTClocation':         'Relative PTC Location',
    'half_life_PC1':               'mRNA Half-Life PC1',
    'cdsseqs_AU_content':          'CDS AU Content',
    'phylop_ptc_to_ejc_median':    'PhyloP (PTC→EJC)',
    'MedianExpression_log2':       'Median Expression (log2)',
    'mut.exon':                    'Mutated Exon',
    'CADD_phred':                  'CADD Score',
    'first.200':                   'PTC in first 200nt',
    'cdsseqs_UC_content':          'CDS UC Content',
    'phastcons_new3utr_first200_median': "PhastCons New 3'UTR first200",
    'AmountExonsAfter':            'Exons Downstream',
    'cdsseq_AUcontentlast200':     'CDS AU Content (last 200)',
    'first.100':                   'PTC in first 100nt',
    'threeUTR_UC_content':         "3'UTR UC Content",
    'isoform_count':               'Isoform Count',
}

## 5. Checks before saving

**Look at the top-12 list.** The manuscript summary plot shows the top 12 features by mean |SHAP|, and its legend is hard-coded (last.EJC colors, "PTC in first 200nt"). If the corrected ranking changed the top 12, the display names and legend in the plotting notebook may need updating.

In [ ]:
mean_abs = np.abs(shap_values_oof).mean(axis=0)
rank = (pd.DataFrame({"feature": X_shap_oof.columns, "mean_abs_shap": mean_abs})
        .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
print("Top 16 features by mean |SHAP| (corrected model):")
print(rank.head(16).to_string(index=False))

top12 = rank.head(12)["feature"].tolist()
print("\nTop-12 features without a display name:", [f for f in top12 if f not in feature_display_names] or "none")
print("Top-12 categorical features:", [f for f in top12 if f in categorical_color_maps or X_shap_oof[f].dtype == object] or "none")
print("Display-name features NOT in top 12:", [f for f in feature_display_names if f not in top12])

# Consistency with notebook 03's ranking file (same sampling => should agree)
rank_csv = OUTPUT_DIR / "shap_feature_importance_rankings.csv"
if rank_csv.exists():
    r3 = pd.read_csv(rank_csv).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    same_top = r3.head(16)["feature"].tolist() == rank.head(16)["feature"].tolist()
    merged = rank.merge(r3, on="feature", suffixes=("_export", "_nb03"))
    print(f"\nnotebook 03 ranking file: top-16 order identical = {same_top}, "
          f"max |diff| in mean|SHAP| = {(merged.mean_abs_shap_export - merged.mean_abs_shap_nb03).abs().max():.2e}")
else:
    print("\n(no notebook 03 ranking file found to compare)")

## 6. Save pickle and manifest

In [ ]:
def _git_commit():
    try:
        return subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return None

shap_data = {
    # --- keys the existing plotting notebooks read (unchanged) ---
    "shap_values": shap_values_oof,
    "feature_data": X_shap_oof,
    "feature_names": X_shap_oof.columns.tolist(),
    "base_value": base_value_avg,
    "categorical_color_maps": categorical_color_maps,
    "feature_display_names": feature_display_names,
    "n_samples": len(X_shap_oof),
    "n_features": X_shap_oof.shape[1],
    "model_info": {
        "n_folds": N_FOLDS,
        "random_state": RANDOM_STATE,
        "notes": "CV-averaged SHAP values from all folds; gene-grouped folds (StratifiedGroupKFold), "
                 "fixed 250 iterations, no early stopping",
    },
    # --- extra keys (ignored by existing notebooks) ---
    "sample_keys": sample_meta["key"].tolist(),      # variant key for each SHAP row
    "sample_meta": sample_meta,                       # key, fold, y_true, oof_prob per SHAP row
    "base_values_per_fold": base_values,
}


with open(PKL_PATH, "wb") as f:
    pickle.dump(shap_data, f)

manifest = {
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "git_commit": _git_commit(),
    "cleaned_data": str(CLEANED_PATH),
    "cv_predictions": str(CV_PRED_PATH),
    "fold_model_files": fold_files,
    "pooled_oof_auc": round(float(oof_auc), 4),
    "n_samples": int(len(X_shap_oof)),
    "samples_per_fold": sample_meta["fold"].value_counts().sort_index().to_dict(),
    "n_features": int(X_shap_oof.shape[1]),
    "shap_sample_size_per_fold": SHAP_SAMPLE_SIZE,
    "random_state": RANDOM_STATE,
    "versions": {"shap": shap.__version__, "catboost": catboost.__version__, "pandas": pd.__version__},
}
with open(OUTPUT_DIR / "shap_export_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

print(f"Saved {PKL_PATH}")
print(f"Saved {OUTPUT_DIR / 'shap_export_manifest.json'}")
print(f"  SHAP values {shap_values_oof.shape}, feature data {X_shap_oof.shape}")